In [ ]:
# Cell 1: Load Cleaned Dataset & Initialize Master Encoded DataFrame
import pandas as pd

input_path = 'data/processed/psu_financial_literacy_cleaned.csv'
df_clean = pd.read_csv(input_path)

df_encoded = pd.DataFrame(index=df_clean.index)
print(f"Loaded cleaned dataset: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")

In [ ]:
# Cell 2: Ordinal, Binary, Perceived Knowledge (q11-q18) & Financial Habits (q26-q42)
ordinal_maps = {
    'q01_age_group': {
        'Prefer not to answer': 0,
        '20-21 years old': 1,
        '22-24 years old': 2,
        '25-27 years old': 3,
        '28 years old or above': 4
    },
    'q05_monthly_allowance': {
        'Prefer not to answer': 0,
        'Below PHP 2,000': 1,
        'PHP 2,000-PHP 4,999': 2,
        'PHP 5,000-PHP 7,999': 3,
        'PHP 8,000-PHP 10,999': 4,
        'PHP 11,000 or above': 5
    },
    'q06_currently_uses_dfs': {'No': 0, 'Yes': 1},
    'q08_dfs_usage_frequency': {
        'Never': 1, 'Rarely': 2, 'Sometimes': 3, 'Often': 4, 'Very often': 5
    },
    'q10_dfs_usage_duration': {
        'I have never used them.': 0,
        'Less than 6 months': 1,
        '6 months to less than 1 year': 2,
        '1-2 years': 3,
        'More than 2 years': 4
    },
    'q43_likelihood_use_bnpl': {
        'I do not use a Buy now, Pay Later (BNPL) Service': 0,
        'Very unlikely': 1, 'Unlikely': 2, 'Neither likely nor unlikely': 3,
        'Likely': 4, 'Very likely': 5
    },
    'q44_likelihood_use_loan': {
        'I do not use any money lending/loaning service': 0,
        'Very unlikely': 1, 'Unlikely': 2, 'Neither likely nor unlikely': 3,
        'Likely': 4, 'Very likely': 5
    },
    'q48_confidence_managing_dfs': {
        'Not at all confident': 1, 'Slightly confident': 2,
        'Moderately confident': 3, 'Very confident': 4, 'Extremely confident': 5
    }
}

for col, mapping in ordinal_maps.items():
    df_encoded[col] = df_clean[col].map(mapping)
    unmapped = df_clean.loc[df_encoded[col].isna(), col].unique()
    if len(unmapped) > 0:
        print(f"[WARNING] Unmapped values in {col}: {unmapped}")

# Explicit dimension lists (reused directly in Cell 7 without positional slicing)
PERC_COLS = [
    'q11_understand_how_dfs_work', 'q12_know_check_tx_and_balance',
    'q13_understand_dfs_fees', 'q14_know_protect_info',
    'q15_recognize_suspicious_scams', 'q16_understand_online_loan_risks',
    'q17_can_compare_dfs', 'q18_understand_check_tx_details'
]

HABIT_COLS = [
    'q26_prepare_budget', 'q27_monitor_expenses', 'q28_save_before_spending',
    'q29_compare_prices', 'q30_consider_affordability',
    'q31_avoid_unnecessary_purchases', 'q32_review_tx_history',
    'q33_check_tx_details', 'q34_check_available_balance',
    'q35_keep_credentials_private', 'q36_verify_suspicious_requests',
    'q37_check_loan_repayment_terms', 'q38_pay_loans_on_time',
    'q39_adjust_spending', 'q40_save_for_emergencies',
    'q41_track_dfs_money_flow', 'q42_decisions_based_on_needs'
]

for col in PERC_COLS + HABIT_COLS:
    df_encoded[col] = df_clean[col].astype(str).str.extract(r'^(\d+)')[0].astype(float)
    if df_encoded[col].isna().any():
        print(f"[WARNING] Missing/unparsed Likert values in {col}")

print("Ordinal, binary, and Likert encoding complete.")

In [ ]:
ANSWER_KEY = {
    'q19_otp_scam_action': 'Never share the OTP and verify the request through official channels.',
    'q20_before_confirming_transfer': 'Check the recipient, amount, and transaction details.',
    'q21_why_check_dfs_fees': 'To understand the cost of using the service.',
    'q22_safest_online_access': 'Use strong, unique passwords and protect account credentials.',
    'q23_phishing_link_action': "Avoid the link and verify the message through the service's official channels.",
    'q24_before_online_loan_bnpl': 'Whether I can meet the repayment terms and afford the total cost.',
    'q25_why_review_tx_history': 'To identify transactions, monitor spending, and notice unfamiliar activity.'
}

for col, correct_answer in ANSWER_KEY.items():
    df_encoded[col] = (df_clean[col] == correct_answer).astype(int)
    print(f"{col}: {df_encoded[col].sum()}/{len(df_encoded)} correct")

In [ ]:
# Explicit option lists ensure unchosen options still exist as 0 columns for future web app inputs
nominal_options = {
    'q02_sex': {
        'q02_male': 'Male',
        'q02_female': 'Female',
        'q02_prefer_not': 'Prefer not to answer'
    },
    'q03_degree_program': {
        'q03_bscs': 'Bachelor of Science in Computer Science',
        'q03_bsit': 'Bachelor of Science in Information Technology',
        'q03_bsmb': 'Bachelor of Science in Marine Biology',
        'q03_bsbio_med': 'Bachelor of Science in Biology Major in Medical Biology',
        'q03_bses': 'Bachelor of Science in Environmental Science'
    },
    'q49_experienced_scam_attempt': {
        'q49_scam_yes': 'Yes',
        'q49_scam_no': 'No',
        'q49_scam_not_sure': 'Not sure',
        'q49_scam_prefer_not': 'Prefer not to answer'
    }
}

for col, options in nominal_options.items():
    matched_any = pd.Series(False, index=df_clean.index)
    for new_col, label in options.items():
        is_match = (df_clean[col] == label)
        df_encoded[new_col] = is_match.astype(int)
        matched_any |= is_match
    unmatched = df_clean.loc[~matched_any, col].unique()
    if len(unmatched) > 0:
        print(f"[WARNING] Unmatched values in {col}: {unmatched}")

print("Nominal one-hot encoding complete.")

In [ ]:
# Cell 5: Multi-Select Checkboxes (q04, q07, q09, q45, q46, q47, q50)
# Uses exact substring matching so internal commas (e.g., q07, q50) never break
multi_select_options = {
    'q04_financial_support_source': {
        'q04_allowance': 'Allowance from parents or guardians',
        'q04_scholarship': 'Scholarship or educational assistance',
        'q04_employment': 'Part-time or full-time employment',
        'q04_personal_savings': 'Personal savings',
        'q04_business': 'Online or small business'
    },
    'q07_dfs_platforms_used': {
        'q07_ewallets': 'E-wallets, such as GCash or Palawan Pay, Atome, Maribank, etc.',
        'q07_mobile_banking': 'Mobile banking applications',
        'q07_online_banking_web': 'Online banking websites',
        'q07_online_payment': 'Online payment services',
        'q07_digital_savings_inv': 'Digital savings or investment platforms',
        'q07_bnpl': 'Buy Now, Pay Later (BNPL) services',
        'q07_online_lending': 'Online lending services',
        'q07_none': 'I have not used any of these services.'
    },
    'q09_dfs_usage_purpose': {
        'q09_send_receive': 'Sending or receiving money',
        'q09_pay_bills': 'Paying bills',
        'q09_buy_goods': 'Buying goods or services',
        'q09_receive_allowance': 'Receiving allowances or salary',
        'q09_save_money': 'Saving money',
        'q09_transfer_accounts': 'Transferring money between accounts',
        'q09_borrow_installment': 'Borrowing money or using installment services',
        'q09_check_balance': 'Checking balances and transaction history'
    },
    'q45_first_action_receiving_money': {
        'q45_save': 'Set aside money for savings.',
        'q45_essentials': 'Pay for essential expenses.',
        'q45_wants': 'Spend on things I want.',
        'q45_divide_planned': 'Divide the money among several planned expenses.',
        'q45_no_approach': 'I do not follow a particular approach.'
    },
    'q46_spending_influences': {
        'q46_personal_needs': 'Personal needs',
        'q46_available_budget': 'Available budget',
        'q46_family_needs': 'Family or household needs',
        'q46_friends_social': 'Friends or social activities',
        'q46_online_ads': 'Online advertisements or social media',
        'q46_discounts_promos': 'Discounts and promotions',
        'q46_digital_convenience': 'Convenience of digital payments'
    },
    'q47_financial_challenges': {
        'q47_limited_allowance': 'Limited allowance or income',
        'q47_diff_saving': 'Difficulty saving money',
        'q47_diff_budgeting': 'Difficulty following a budget',
        'q47_unexpected_expenses': 'Unexpected expenses',
        'q47_impulse_buying': 'Impulse buying or unplanned spending',
        'q47_limited_dfs_knowledge': 'Limited knowledge of digital financial services',
        'q47_scam_concern': 'Concern about online scams or fraud',
        'q47_diff_fees_terms': 'Difficulty understanding fees or repayment terms',
        'q47_none': 'I do not experience these challenges.'
    },
    'q50_areas_to_improve': {
        'q50_budgeting': 'Budgeting',
        'q50_saving': 'Saving',
        'q50_digital_payments': 'Managing digital payments',
        'q50_account_protection': 'Protecting financial accounts',
        'q50_recognizing_scams': 'Recognizing online financial scams',
        'q50_loans_interest_fees': 'Understanding digital loans, interest, and fees',
        'q50_future_planning': 'Planning for future financial needs'
    }
}

# Collect all new binary columns in a dictionary first to avoid DataFrame fragmentation
new_cols = {}

for col, options in multi_select_options.items():
    leftover = df_clean[col].fillna('')
    for new_col, text in options.items():
        new_cols[new_col] = df_clean[col].str.contains(text, regex=False, na=False).astype(int)
        leftover = leftover.str.replace(text, '', regex=False)
    
    # Keep custom "Other:" responses faithfully flagged in their respective *_other column
    leftover = leftover.str.replace(r'^[\s,]+|[\s,]+$', '', regex=True)
    other_col = col.split('_')[0] + '_other'
    new_cols[other_col] = (leftover != '').astype(int)
    
    custom_entries = leftover[leftover != '']
    if len(custom_entries) > 0:
        print(f"=== Custom 'Other' entries in {col} ({len(custom_entries)}) ===")
        print(custom_entries.value_counts(), "\n")

df_encoded = pd.concat([df_encoded, pd.DataFrame(new_cols, index=df_clean.index)], axis=1)
print("Multi-select encoding complete.")

In [ ]:
# Cell 6: Pre-Target Feature Integrity Check & Integer Casting (No Export Yet)
nan_counts = df_encoded.isnull().sum()
cols_with_nans = nan_counts[nan_counts > 0]

print(f"Encoded feature matrix shape: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns")
print(f"Missing feature values: {df_encoded.isnull().sum().sum()}")

if len(cols_with_nans) > 0:
    print("\n[ACTION NEEDED] Columns with NaNs:")
    print(cols_with_nans)
else:
    df_encoded = df_encoded.astype(int)
    print("All encoded feature columns verified and cast to integer.")

In [ ]:
# Cell 7: Target Variable Construction (True 0-100 Min-Max + OECD/INFE 70/100) & Final Export
OBJ_COLS = list(ANSWER_KEY.keys())  # q19-q25 (7 binary items, 0-1)

# 1. True 0-100 Min-Max Rescaling per dimension
obj_pct = (df_encoded[OBJ_COLS].sum(axis=1) / len(OBJ_COLS)) * 100
perc_pct = ((df_encoded[PERC_COLS].sum(axis=1) - len(PERC_COLS)) / (len(PERC_COLS) * 4)) * 100
habit_pct = ((df_encoded[HABIT_COLS].sum(axis=1) - len(HABIT_COLS)) / (len(HABIT_COLS) * 4)) * 100

df_encoded['dfl_score'] = ((obj_pct + perc_pct + habit_pct) / 3).round(2)

# 2. Strict OECD/INFE (2026) Minimum Target Score (70 out of 100)
OECD_THRESHOLD = 70.0
df_encoded['dfl_class'] = (df_encoded['dfl_score'] >= OECD_THRESHOLD).astype(int)

# 3. Verification summary
print("=== True 0-100 Composite DFL Score Summary ===")
print(df_encoded['dfl_score'].describe().round(2))

print("\n=== OECD/INFE 70/100 Target Class Distribution ===")
class_counts = df_encoded['dfl_class'].value_counts().sort_index()
class_pcts = (df_encoded['dfl_class'].value_counts(normalize=True).sort_index() * 100).round(2)
summary_df = pd.DataFrame({
    'Label': ['Below Target (<70)', 'Meets Target (>=70)'],
    'Count': class_counts.values,
    'Percent (%)': class_pcts.values
}, index=class_counts.index)
print(summary_df)

# 4. Single Final Export
total_nans = df_encoded.isnull().sum().sum()
print(f"\nFinal shape: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns")
print(f"Total missing values: {total_nans}")

if total_nans == 0:
    output_path = 'data/processed/psu_financial_literacy_encoded.csv'
    df_encoded.to_csv(output_path, index=False)
    print(f"Saved finalized encoded dataset to: {output_path}")
else:
    print("[ACTION NEEDED] Missing values detected—export halted.")